# Demonstration of Noisy Expectation-Maximization (NEM)

This notebook demonstrates the behavior of the NEM algorithms implemented in the `py_nem` library. We will run the algorithms with varying amounts of noise and observe the effect on convergence speed.

## 1. GMM-NEM Demonstration

First, we will test the `gmm_nem_nd` algorithm. We will generate synthetic 2D data from a known Gaussian Mixture Model and then run the NEM algorithm for a range of `noise_sigma` values. We will plot the number of iterations required for convergence against the noise sigma.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from py_nem import em
from py_nem import clustering

# for reproducibility
np.random.seed(42)

In [ ]:
# Generate synthetic 2D GMM data
true_mus = np.array([[-3, -3], [3, 3]])
true_sigmas = np.array([[[1, 0.5], [0.5, 1]], [[1, -0.5], [-0.5, 1]]])
true_weights = np.array([0.5, 0.5])
n_samples = 1000
n_components = 2
n_dims = 2

n0 = int(n_samples * true_weights[0])
n1 = n_samples - n0
data_0 = np.random.multivariate_normal(true_mus[0], true_sigmas[0], n0)
data_1 = np.random.multivariate_normal(true_mus[1], true_sigmas[1], n1)
data = np.concatenate([data_0, data_1])

print(f"Generated {n_samples} data points in {n_dims} dimensions.")

In [ ]:
# Define the range of noise sigmas to test
noise_sigmas = np.linspace(0, 1.0, 20)
convergence_steps_gmm = []

print("Running GMM-NEM sweep...")
for sigma in noise_sigmas:
    initial_mus = np.array([[-1, -1], [1, 1]])
    initial_sigmas = np.array([np.eye(2), np.eye(2)])
    initial_weights = np.array([0.5, 0.5])
    params_history = em.gmm_nem_nd(data, initial_mus, initial_sigmas, initial_weights, noise_sigma=sigma, tol=1e-4, max_iter=200)
    steps = len(params_history) - 1
    convergence_steps_gmm.append(steps)
print("GMM Sweep complete.")

## 2. K-Means-NEM Demonstration

In [ ]:
noise_sigmas_kmeans = np.linspace(0, 1.0, 20)
convergence_steps_kmeans = []

print("\nRunning K-Means-NEM sweep...")
for sigma in noise_sigmas_kmeans:
    initial_centroids = np.array([[-1.0, -1.0], [1.0, 1.0]])
    labels, history = clustering.kmeans_nem(data, n_clusters=n_components, initial_centroids=initial_centroids, noise_sigma=sigma, max_iter=200)
    steps = len(history) - 1
    convergence_steps_kmeans.append(steps)
print("K-Means Sweep complete.")

## 3. Censored Gamma-NEM Demonstration

In [ ]:
# Generate synthetic Gamma data
true_alpha = 2.0
true_theta = 1.5
n_samples_gamma = 2000

gamma_data = np.random.gamma(shape=true_alpha, scale=true_theta, size=n_samples_gamma)

# Censor the data
T = np.percentile(gamma_data, 80)
gamma_data[gamma_data > T] = T

print(f"Generated {n_samples_gamma} censored Gamma data points.")

In [ ]:
noise_sigmas_gamma = np.linspace(0, 0.5, 20)
convergence_steps_gamma = []

print("\nRunning Gamma-NEM sweep...")
for sigma in noise_sigmas_gamma:
    initial_theta = 1.0
    theta_history = em.gamma_nem_censored(gamma_data, alpha=true_alpha, T=T, initial_theta=initial_theta, noise_sigma=sigma, max_iter=200)
    steps = len(theta_history) - 1
    convergence_steps_gamma.append(steps)
print("Gamma Sweep complete.")

## 4. Results

In [ ]:
plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
plt.plot(noise_sigmas, convergence_steps_gmm, 'o-')
plt.xlabel("Noise Sigma")
plt.ylabel("Steps to Converge")
plt.title("GMM-NEM Convergence")
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(noise_sigmas_kmeans, convergence_steps_kmeans, 'o-', color='orange')
plt.xlabel("Noise Sigma")
plt.ylabel("Steps to Converge")
plt.title("K-Means-NEM Convergence")
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(noise_sigmas_gamma, convergence_steps_gamma, 'o-', color='green')
plt.xlabel("Noise Sigma")
plt.ylabel("Steps to Converge")
plt.title("Gamma-NEM Convergence")
plt.grid(True)

plt.tight_layout()
plt.show()